# RAG Evaluation
### Ein wissenschaftliches Notebook zur Evaluation von Retrieval-Augmented-Generation-Systemen


---

## Einordnung in die Plattform

Dieses Notebook evaluiert die Pipeline aus den Notebooks 01 bis 05. Es verwendet die
**echten Metrik-Implementierungen** des Pakets `rag.evaluation`. Dabei greift es auf
beide Ebenen des Metrik-Pakets zu:

* die **Prediction-Level-Metriken** (`ContextPrecisionMetric`, `ContextRecallMetric`,
  `MRRMetric`, `NDCGMetric`, `ExactMatchMetric`, `TokenF1Metric`, `LatencyMetric`,
  `ThroughputMetric`, `MemoryUsageMetric`), die über den `EvaluationRunner` laufen, und
* die **funktionalen Metrik-Primitive** aus `rag.evaluation.metrics.functional`
  (`precision_at_k`, `recall_at_k`, `hit_at_k`, `context_overlap`), die wir für die
  Einzelbeispiel-Demonstrationen direkt importieren.

Es gibt damit **eine einzige Quelle** für jede Metrik: das Paket. Lediglich die n-gramm-
und Ähnlichkeitsmetriken **BLEU, ROUGE-1, ROUGE-L** und eine **lexikalische
Ähnlichkeits-Näherung** werden im Notebook didaktisch implementiert, mit ihrer
mathematischen Definition, weil das Paket sie bewusst nicht enthält. Ihre Definition
sehen zu können, ist hier der didaktische Zweck.

Das Notebook ist **vollständig eigenständig lauffähig**: Es benötigt weder GPU noch
Ollama noch Modell-Downloads. Wir konstruieren einen kleinen, nachvollziehbaren
Benchmark-Datensatz mit Ground Truth und einen transparenten Demonstrations-Retriever,
sodass jede Metrik an einem Fall beobachtbar wird, dessen korrekte Antwort wir bereits
kennen.

---

### Warum ein eigenes Evaluationsnotebook?

Die Pipeline-Notebooks 01 bis 05 enthalten Bereitschaftsprüfungen („funktioniert die Stufe
überhaupt?"). Evaluation ist etwas anderes: Sie fragt **wie gut** das System ist, und das
ist eine wissenschaftliche, nicht nur eine technische Frage. Sie verlangt einen
Goldstandard, definierte Metriken, kontrollierte Experimente und eine ehrliche
Interpretation, inklusive der Grenzen der Metriken selbst.

---

## 0. Imports und reproduzierbarer Kontext

Alle Zufallsquellen werden geseedet. Reproduzierbarkeit ist in der Evaluation keine
Annehmlichkeit, sondern Voraussetzung: eine Metrik, die bei jedem Lauf schwankt, ist
keine Messung, sondern Rauschen (siehe Abschnitt 14).

In [ ]:
import math
import re
import time
import random
import logging
from collections import Counter
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

random.seed(42)
np.random.seed(42)

for name in ["rag.evaluation", "rag.retrieval", "rag.indexing"]:
    logging.getLogger(name).setLevel(logging.ERROR)

matplotlib.rcParams["figure.dpi"] = 110
matplotlib.rcParams["axes.grid"] = True
matplotlib.rcParams["grid.alpha"] = 0.25
matplotlib.rcParams["font.size"] = 10

print("Imports complete.")

---

## 1. Warum RAG-Evaluation schwierig ist

Ein RAG-System zu evaluieren ist deutlich schwerer als ein Klassifikationsmodell. Die
Gründe sind strukturell, nicht handwerklich.

**Zwei verkettete Subsysteme.** Ein RAG-System ist Retrieval *und* Generation. Eine
schlechte Endantwort kann zwei völlig verschiedene Ursachen haben: der Retriever fand die
relevanten Dokumente nicht, oder der Generator nutzte gefundene Dokumente schlecht. Eine
Metrik auf der Endantwort allein kann diese Fälle nicht trennen. Deshalb evaluiert man
Retrieval und Generation **getrennt** (Abschnitt 4).

**Keine eindeutig richtige Antwort.** Bei Generierung gibt es selten *die eine* korrekte
Formulierung. „Mistral 7B hat 7 Milliarden Parameter" und „Das Modell besitzt 7B Parameter"
sind beide korrekt, aber Exact Match wertet das zweite als Fehler. Oberflächenmetriken
strafen legitime Variation ab (Abschnitt 7).

**Ground Truth ist teuer und unscharf.** Relevanzurteile sind subjektiv. Ist ein Dokument,
das die Frage *teilweise* beantwortet, relevant? Annotatoren widersprechen sich. Jeder
Benchmark erbt die Verzerrungen seiner Annotation (Abschnitt 14).

**Halluzination ist schwer maschinell zu messen.** Eine flüssige, selbstbewusste, aber
faktisch falsche Antwort ist das gefährlichste Fehlermuster, und genau das, was
n-gramm-basierte Metriken am schlechtesten erkennen (Abschnitt 12).

**Der Evaluator wird selbst zum Modell.** LLM-as-a-Judge ist verbreitet, aber der Richter
erbt die Schwächen eines LLM: Verzerrungen, Inkonsistenz, Anfälligkeit für Formulierung
(Abschnitt 8).

Die Konsequenz: Es gibt **keine** Einzahl, die „RAG-Qualität" misst. Es gibt ein Bündel
von Metriken, von denen jede einen Aspekt beleuchtet und jede eigene blinde Flecken hat.
Professionelle Evaluation besteht darin, dieses Bündel zu verstehen, inklusive der Frage,
welche Metrik wann *irreführt*.


---

## 2. Architektur einer Evaluationspipeline

Das Paket `rag.evaluation` strukturiert Evaluation entlang klarer, typisierter Verträge.

```text
EvaluationExample            (Query + expected_answer + relevant_document_ids)
       │
       ▼
EvaluationRunner.run(examples)
       │   für jedes Beispiel:
       │     Retriever.retrieve(query, k)  → RetrievedContext[]
       │     Strategy.generate(query, ctx) → GeneratedAnswer        (optional)
       │     PerformanceMonitor             → StageTiming, Momentaufnahmen
       ▼
EvaluationPrediction[]       (example + retrieved + generated + timings)
       │
       ▼   jede Metrik:  BaseMetric.evaluate(predictions) → MetricResult
ContextPrecision · ContextRecall · MRR · nDCG · ExactMatch · TokenF1 · Latency · ...
       │
       ▼
EvaluationRunResult  →  EvaluationReport.summary()  →  JSONL/CSV-Export
```

**Drei Modi.** Der `EvaluationRunner` kennt `"retrieval"` (kein LLM), `"generation"`
(kein Retrieval) und `"end_to_end"` (beides). Das macht die Trennung aus Abschnitt 1
operationalisierbar: Retrieval-Qualität wird ohne den verrauschenden Einfluss der
Generierung gemessen.

**Stateless-Metriken.** Jede Metrik implementiert `BaseMetric` und ist zustandslos:
`evaluate(predictions)` liefert für dieselben Eingaben immer dasselbe Ergebnis. Das ist
die Grundlage für Reproduzierbarkeit.

**Fehler werden eingefangen, nicht geworfen.** Pro-Beispiel-Fehler landen in
`EvaluationPrediction.errors`, statt den ganzen Lauf abzubrechen. Ein einzelnes
fehlgeschlagenes Beispiel verfälscht so keine ganze Messreihe.


---

## 3. Der Benchmark-Datensatz

Wir bauen einen kleinen, vollständig kontrollierten Korpus und einen dazu passenden
Goldstandard. „Kontrolliert" heißt: Wir wissen für jede Query, welche Dokumente relevant
sind, denn nur dann sind Retrieval-Metriken überhaupt berechenbar.

Der Korpus deckt ein eng umrissenes Themenfeld (RAG/IR/ML) ab, mit teils überlappenden
Begriffen. Diese Überlappung ist beabsichtigt: Sie erzeugt realistische
Verwechslungsfälle, an denen sich Metriken differenzieren lassen.

In [ ]:
# Controlled corpus: doc_id -> text. Topics overlap deliberately.
CORPUS = {
    "doc-rag":      "Retrieval-Augmented Generation kombiniert einen Retriever mit einem Sprachmodell. Der Retriever findet relevante Dokumente, das Sprachmodell formuliert die Antwort auf deren Basis.",
    "doc-embed":    "Ein Embedding-Modell projiziert Text in einen dichten Vektorraum. Semantisch aehnliche Texte liegen dort nahe beieinander, unabhaengig von den verwendeten Woertern.",
    "doc-bm25":     "BM25 ist ein probabilistisches Rankingmodell. Es verwendet TF-Saettigung ueber den Parameter k1 und Laengennormalisierung ueber den Parameter b.",
    "doc-faiss":    "FAISS ist eine Bibliothek fuer effiziente Nearest-Neighbor-Suche. Sie unterstuetzt exakte Flat-Indizes und approximierte Strukturen wie HNSW und IVF.",
    "doc-cosine":   "Cosine-Similaritaet misst den Winkel zwischen zwei Vektoren. Bei L2-normalisierten Vektoren ist sie identisch mit dem Skalarprodukt.",
    "doc-chunk":    "Chunking teilt Dokumente in kleinere Einheiten. Zu grosse Chunks erzeugen unscharfe Vektoren, zu kleine verlieren Kontext.",
    "doc-hybrid":   "Hybrid-Retrieval fusioniert dichte und lexikalische Signale. Reciprocal Rank Fusion kombiniert Ergebnislisten allein anhand der Raenge.",
    "doc-mrr":      "Mean Reciprocal Rank misst, wie weit oben das erste relevante Dokument steht. Der Kehrwert seines Rangs wird ueber alle Queries gemittelt.",
    "doc-ndcg":     "Normalized Discounted Cumulative Gain bewertet die gesamte Rangliste und gewichtet fruehe Positionen staerker ueber einen logarithmischen Discount.",
    "doc-halluz":   "Halluzination bezeichnet faktisch falsche, aber fluessig formulierte Modellausgaben. Sie ist im RAG-Kontext besonders gefaehrlich, wenn der Kontext die Antwort nicht stuetzt.",
    "doc-temp":     "Die Temperature steuert die Zufaelligkeit der Generierung. Bei temperature gleich null waehlt das Modell stets das wahrscheinlichste Token (greedy decoding).",
    "doc-rerank":   "Reranking ordnet Retrieval-Kandidaten mit einem zweiten, teureren Modell neu. Es verbessert die Praezision der oberen Raenge.",
    "doc-eval":     "Evaluation eines RAG-Systems trennt Retrieval- und Generation-Qualitaet. Eine schlechte Endantwort kann an beidem liegen.",
    "doc-vram":     "VRAM ist der Speicher der GPU. Er limitiert Batch-Groesse und Modellwahl. Reduzierte Praezision wie fp16 halbiert den Bedarf.",
}
print(f"Korpus: {len(CORPUS)} Dokumente")
for did, txt in list(CORPUS.items())[:3]:
    print(f"  {did:<12} {txt[:60]}...")

In [ ]:
from rag.evaluation.dataset import dataset_from_dicts

# Gold standard: query + expected_answer + relevant_document_ids.
GOLD = [
    {"example_id": "q1", "query": "Was kombiniert Retrieval-Augmented Generation?",
     "expected_answer": "Einen Retriever mit einem Sprachmodell.",
     "relevant_document_ids": ["doc-rag"]},
    {"example_id": "q2", "query": "Welche zwei Parameter steuern BM25?",
     "expected_answer": "k1 und b.",
     "relevant_document_ids": ["doc-bm25"]},
    {"example_id": "q3", "query": "Was misst Mean Reciprocal Rank?",
     "expected_answer": "Wie weit oben das erste relevante Dokument steht.",
     "relevant_document_ids": ["doc-mrr"]},
    {"example_id": "q4", "query": "Wann ist Cosine-Similaritaet gleich dem Skalarprodukt?",
     "expected_answer": "Bei L2-normalisierten Vektoren.",
     "relevant_document_ids": ["doc-cosine"]},
    {"example_id": "q5", "query": "Was bewertet nDCG und wie gewichtet es Raenge?",
     "expected_answer": "Die gesamte Rangliste, mit logarithmischem Discount fuer spaetere Positionen.",
     "relevant_document_ids": ["doc-ndcg"]},
    {"example_id": "q6", "query": "Warum ist Halluzination im RAG-Kontext gefaehrlich?",
     "expected_answer": "Weil die Antwort fluessig, aber faktisch falsch ist und vom Kontext nicht gestuetzt wird.",
     "relevant_document_ids": ["doc-halluz"]},
    {"example_id": "q7", "query": "Was bedeutet temperature gleich null bei der Generierung?",
     "expected_answer": "Greedy decoding: das wahrscheinlichste Token wird stets gewaehlt.",
     "relevant_document_ids": ["doc-temp"]},
    {"example_id": "q8", "query": "Was tut Reranking?",
     "expected_answer": "Es ordnet Kandidaten mit einem zweiten Modell neu und verbessert die Praezision.",
     "relevant_document_ids": ["doc-rerank"]},
    # q9 is deliberately hard: the wording shares almost no content words with its
    # relevant document (doc-faiss talks about "Nearest-Neighbor-Suche", "Flat", "HNSW",
    # "IVF" - none of which appear here). A purely lexical retriever must struggle, which
    # gives the Failure-Case section (12) a genuine failure to dissect.
    {"example_id": "q9", "query": "Welches Werkzeug findet schnell die aehnlichsten Eintraege in einer riesigen Sammlung?",
     "expected_answer": "Die FAISS-Bibliothek.",
     "relevant_document_ids": ["doc-faiss"]},
]

DATASET = dataset_from_dicts(GOLD)
print(f"Benchmark-Datensatz: {len(DATASET)} Beispiele")
print()
ex = DATASET[0]
print("Struktur eines EvaluationExample:")
print(f"  example_id            : {ex.example_id}")
print(f"  query                 : {ex.query}")
print(f"  expected_answer       : {ex.expected_answer}")
print(f"  relevant_document_ids : {ex.relevant_document_ids}")

**Hinweis zur Größe.** Neun Beispiele sind für eine *belastbare* Messung zu wenig. Der
`EvaluationReport` warnt zu Recht unterhalb von zehn Beispielen. Für ein didaktisches
Notebook ist das richtig: Wir wollen jede Metrik an wenigen, durchschaubaren Fällen
beobachten, nicht statistische Signifikanz behaupten. Eines der Beispiele (q9) ist
bewusst schwer formuliert, damit später ein echter Fehlerfall zu sehen ist.
Abschnitt 14 vertieft, warum kleine Benchmarks irreführen.

---

## 4. Retrieval- vs. Generation-Evaluation

Die wichtigste konzeptionelle Trennung. Ein RAG-System hat zwei Qualitätsachsen, die man
**nie** vermischen darf:

**Retrieval-Evaluation** fragt: *Hat das System die relevanten Dokumente gefunden, und
weit genug oben?* Sie braucht nur die Rangliste und die Ground-Truth-Relevanz, kein LLM.
Metriken: Precision@k, Recall@k, MRR, nDCG, Hit Rate.

**Generation-Evaluation** fragt: *Ist die formulierte Antwort korrekt, vollständig und vom
Kontext gestützt?* Sie braucht die generierte Antwort und eine Referenz. Metriken: Exact
Match, Token-F1, semantische Ähnlichkeit, BLEU/ROUGE, Faithfulness, Groundedness.

Warum die Trennung zwingend ist, zeigt eine 2×2-Fehlermatrix:

| | Generator gut | Generator schlecht |
|---|---|---|
| **Retrieval gut** | ideale Antwort | Generator-Problem (Kontext war da, wurde verschenkt) |
| **Retrieval schlecht** | Halluzination oder Glück | Totalausfall |

Nur getrennte Metriken erlauben, eine schlechte Endantwort der richtigen Stufe
zuzuordnen. Wer nur die Endantwort misst, optimiert blind.

---

## 5. Offline- vs. Online-Evaluation

**Offline-Evaluation** läuft gegen einen festen, annotierten Datensatz, wie in diesem
Notebook. Sie ist reproduzierbar, billig, schnell und erlaubt fairen Vergleich zweier
Systemvarianten unter identischen Bedingungen. Ihre Schwäche: Sie misst nur, was im
Datensatz steht. Verteilungsverschiebungen der echten Nutzerfragen sieht sie nicht.

**Online-Evaluation** misst am laufenden System mit echten Nutzern: Klickraten,
Daumen-hoch/runter, Aufgabenerfolg, A/B-Tests. Sie misst, was wirklich zählt, ist aber
teuer, langsam, verrauscht und ethisch heikel (man experimentiert an Nutzern).

In der Praxis ist die Reihenfolge fest: **offline filtern, online bestätigen.** Offline
verwirft man die offensichtlich schlechteren Varianten billig; nur die wenigen
verbleibenden Kandidaten kommen in den teuren Online-Test. Dieses Notebook adressiert die
offline Hälfte, die Grundlage jeder seriösen Systementscheidung.

---

## 6. Retrieval-Metriken: Mathematik und Implementierung

Wir definieren jede Metrik formal, interpretieren sie intuitiv und demonstrieren sie an
einem kontrollierten Experiment. Notation: $R$ = Menge der relevanten Dokumente einer
Query, $\text{ret}_k$ = die ersten $k$ zurückgegebenen Dokumente.

### Precision@k und Recall@k

$$\text{Precision@}k = \frac{|R \cap \text{ret}_k|}{k}
\qquad
\text{Recall@}k = \frac{|R \cap \text{ret}_k|}{|R|}$$

**Precision@k** beantwortet: *Wie viel vom Gezeigten ist relevant?*, wichtig, wenn der
Generator nur wenige Chunks verarbeiten kann. **Recall@k** beantwortet: *Wie viel vom
Relevanten wurde gefunden?*, wichtig, wenn nichts Wesentliches fehlen darf. Die beiden
stehen im klassischen Zielkonflikt: größeres $k$ erhöht Recall, senkt tendenziell
Precision.

### Hit Rate@k

$$\text{HitRate@}k = \frac{1}{|Q|}\sum_{q\in Q} \mathbb{1}\!\left[|R_q \cap \text{ret}_k^q| > 0\right]$$

Der Anteil der Queries, für die *mindestens ein* relevantes Dokument unter den Top-$k$
ist. Eine grobe, aber in der Praxis sehr aussagekräftige Metrik: Wenn die Hit Rate niedrig
ist, kann der Generator gar nicht gewinnen.

### Mean Reciprocal Rank (MRR)

$$\text{RR}(q) = \frac{1}{\text{rank of first relevant}}, \qquad
\text{MRR} = \frac{1}{|Q|}\sum_{q\in Q}\text{RR}(q)$$

Belohnt es, das erste relevante Dokument *weit oben* zu platzieren. Rang 1 → 1.0, Rang 2 →
0.5, Rang 4 → 0.25. Ignoriert alles nach dem ersten Treffer.

### Normalized Discounted Cumulative Gain (nDCG@k)

$$\text{DCG@}k = \sum_{i=1}^{k}\frac{rel_i}{\log_2(i+1)}, \qquad
\text{nDCG@}k = \frac{\text{DCG@}k}{\text{IDCG@}k}$$

Bewertet die *gesamte* Rangliste mit logarithmisch fallendem Gewicht und normiert auf die
ideale Reihenfolge (IDCG). Der umfassendste Ranking-Indikator: anders als MRR zählen auch
spätere Treffer.

In [ ]:
# Retrieval-IR-Primitive kommen aus dem Paket-– eine einzige Quelle der Wahrheit.
# (Definitionen siehe Markdown oben; Implementierung in rag.evaluation.metrics.functional.)
from rag.evaluation.metrics.functional import precision_at_k, recall_at_k, hit_at_k

# Quick sanity demo on a known ranking.
ranking = ["doc-bm25", "doc-faiss", "doc-cosine", "doc-rag"]
relevant_set = {"doc-cosine"}
for k in (1, 2, 3, 4):
    print(f"k={k}  P@k={precision_at_k(ranking, relevant_set, k):.3f}  "
          f"R@k={recall_at_k(ranking, relevant_set, k):.3f}  "
          f"Hit@k={int(hit_at_k(ranking, relevant_set, k))}")
print("\nDas relevante Dokument steht auf Rang 3: P@3=1/3, R@3=1.0, Hit@3=1.")

### Demonstrations-Retriever

Wir brauchen Ranglisten, um die Metriken zu füttern. In Produktion liefert sie der
`RetrievalOrchestrator` (Notebook 04). Für ein eigenständiges Notebook verwenden wir einen
**transparenten Token-Overlap-Retriever**: Er rankt Dokumente nach lexikalischer
Überlappung mit der Query. Bewusst simpel, so ist jedes Ranking nachvollziehbar, und wir
erhalten einen realistischen Mix aus guten und schlechten Treffern für die
Fehleranalyse. Er liefert exakt die Dict-Form, die der `EvaluationRunner` erwartet
(`document_id`, `chunk_id`, `text`, `score`).

In [ ]:
_WORD = re.compile(r"\b\w+\b")

def tokenize(text: str) -> List[str]:
    return _WORD.findall(text.lower())

class TokenOverlapRetriever:
    """Didactic retriever: ranks corpus docs by query/term overlap.

    Returns the dict shape consumed by EvaluationRunner:
    {document_id, chunk_id, text, score}. In production this is the
    RetrievalOrchestrator (dense + sparse + fusion).
    """
    def __init__(self, corpus: Dict[str, str]):
        self.corpus = corpus
        self.doc_tokens = {d: Counter(tokenize(t)) for d, t in corpus.items()}

    def retrieve(self, query: str, k: int = 10) -> List[dict]:
        q = Counter(tokenize(query))
        scored = []
        for did, toks in self.doc_tokens.items():
            overlap = sum(min(q[w], toks[w]) for w in q)
            norm = math.sqrt(sum(v*v for v in q.values())) * math.sqrt(sum(v*v for v in toks.values()))
            score = overlap / norm if norm else 0.0
            scored.append((did, score))
        scored.sort(key=lambda x: (-x[1], x[0]))
        out = []
        for did, score in scored[:k]:
            out.append({
                "document_id": did,
                "chunk_id": did,          # one chunk per doc in this corpus
                "text": self.corpus[did],
                "score": float(score),
            })
        return out

retriever = TokenOverlapRetriever(CORPUS)
demo = retriever.retrieve(DATASET[0].query, k=4)
print(f"Query: {DATASET[0].query!r}")
for r in demo:
    print(f"  {r['document_id']:<12} score={r['score']:.3f}")

### Kontrolliertes Experiment: perfektes vs. degradiertes Ranking

Metriken versteht man am besten, wenn man ihr Verhalten bei *bekannter* Eingabe sieht. Wir
vergleichen drei synthetische Ranglisten gegen eine einzelne relevante Position und
beobachten, wie die Metriken reagieren.

In [ ]:
from rag.evaluation.types import EvaluationExample, EvaluationPrediction, RetrievedContext
from rag.evaluation.metrics.mrr import MRRMetric
from rag.evaluation.metrics.ndcg import NDCGMetric

def make_prediction(example: EvaluationExample, ranked_ids: List[str]) -> EvaluationPrediction:
    ctxs = [RetrievedContext(document_id=d, chunk_id=d, text=CORPUS.get(d, ""),
                             score=1.0 - i*0.01, rank=i)
            for i, d in enumerate(ranked_ids)]
    return EvaluationPrediction(example=example, retrieved_contexts=ctxs)

probe = EvaluationExample(query="probe", expected_answer="x", relevant_document_ids=["doc-cosine"])

rankings = {
    "perfekt   (Rang 1)": ["doc-cosine", "doc-bm25", "doc-faiss", "doc-rag"],
    "mittel    (Rang 3)": ["doc-bm25", "doc-faiss", "doc-cosine", "doc-rag"],
    "schlecht  (nicht in Top-4)": ["doc-bm25", "doc-faiss", "doc-rag", "doc-embed"],
}

mrr = MRRMetric()
ndcg = NDCGMetric(k=4)

print(f"{'Ranking':<28} {'P@1':>5} {'P@3':>5} {'R@3':>5} {'MRR':>6} {'nDCG@4':>7}")
print("-" * 62)
for label, ids in rankings.items():
    pred = make_prediction(probe, ids)
    rel = set(probe.relevant_document_ids)
    p1 = precision_at_k(ids, rel, 1)
    p3 = precision_at_k(ids, rel, 3)
    r3 = recall_at_k(ids, rel, 3)
    m = mrr.evaluate([pred]).value
    n = ndcg.evaluate([pred]).value
    print(f"{label:<28} {p1:>5.2f} {p3:>5.2f} {r3:>5.2f} {m:>6.3f} {n:>7.3f}")

**Interpretation.** Beim perfekten Ranking sind MRR und nDCG@4 = 1.0. Rutscht das
relevante Dokument auf Rang 3, fällt MRR auf 0.33 und nDCG sinkt durch den
log-Discount. Fehlt es ganz, sind beide 0. P@1 ist binär (1 oder 0) und damit grob; P@3
mittelt über drei Positionen. So sieht man konkret: MRR bestraft tiefe erste Treffer hart,
nDCG bewertet die Gesamtstruktur, Precision@k hängt empfindlich von $k$ ab.

In [ ]:
# Precision/Recall over k, averaged across the real benchmark with the demo retriever.
ks = [1, 2, 3, 5, 8, 10]
mean_p, mean_r, mean_hit = [], [], []
for k in ks:
    ps, rs, hs = [], [], []
    for ex in DATASET:
        ids = [r["document_id"] for r in retriever.retrieve(ex.query, k=k)]
        rel = set(ex.relevant_document_ids)
        ps.append(precision_at_k(ids, rel, k))
        rs.append(recall_at_k(ids, rel, k))
        hs.append(hit_at_k(ids, rel, k))
    mean_p.append(np.mean(ps)); mean_r.append(np.mean(rs)); mean_hit.append(np.mean(hs))

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(ks, mean_p, marker="o", label="Precision@k", color="#2563eb")
ax.plot(ks, mean_r, marker="s", label="Recall@k", color="#16a34a")
ax.plot(ks, mean_hit, marker="^", label="Hit Rate@k", color="#dc2626")
ax.set_xlabel("k"); ax.set_ylabel("Wert (gemittelt ueber Benchmark)")
ax.set_title("Precision / Recall / Hit Rate ueber k")
ax.set_ylim(-0.02, 1.05); ax.legend()
plt.tight_layout(); plt.show()
print("Recall steigt monoton mit k; Precision faellt typischerweise, weil mehr")
print("irrelevante Dokumente mit aufgenommen werden. Das ist der Precision/Recall-Tradeoff.")

### Context Precision und Context Recall aus dem Paket

Das `rag.evaluation`-Paket implementiert die ranking-bezogenen Größen als **Context
Precision** und **Context Recall** über alle Beispiele. Wir wenden sie auf echte
Ranglisten des Demonstrations-Retrievers an, und zwar über den `EvaluationRunner`.

In [ ]:
from rag.evaluation.config import EvaluationConfig
from rag.evaluation.runner import EvaluationRunner
from rag.evaluation.report import EvaluationReport

retrieval_config = EvaluationConfig(
    mode="retrieval",
    top_k=5,
    retrieval_metrics=["context_precision", "context_recall", "mrr", "ndcg"],
    generation_metrics=[],
    performance_metrics=["latency", "throughput"],
    capture_resources=True,
)

runner = EvaluationRunner(config=retrieval_config, retriever=retriever)
retrieval_run = runner.run(DATASET)

print(EvaluationReport(retrieval_run).summary())

**Interpretation.** Der Report bündelt alle aktivierten Metriken in einer Tabelle und
hängt automatisch Warnungen an, hier die Warnung, dass weniger als zehn Beispiele für
belastbare Schlüsse zu wenig sind. Genau diese eingebaute Skepsis unterscheidet eine
Evaluationsbibliothek von einer bloßen Metrik-Sammlung.

Ein Wert verdient Einordnung: **Context Precision** wirkt mit ~0,2 niedrig, ist hier aber
strukturell gedeckelt. Bei `top_k=5` und genau **einem** relevanten Dokument pro Query
kann höchstens 1 von 5 zurückgegebenen Dokumenten relevant sein. Die Obergrenze ist also
1/5 = 0,2. Dass MRR und nDCG zugleich hoch sind, zeigt: das relevante Dokument steht
meist ganz oben; die übrigen vier Plätze sind notwendigerweise mit (für diese Query)
irrelevanten Dokumenten gefüllt. Precision@k ist deshalb nur im Verhältnis zur Zahl der
relevanten Dokumente interpretierbar.

---

## 7. Generation-Metriken: Mathematik und Implementierung

Jetzt die zweite Achse: Wie gut ist die *formulierte* Antwort? Wir verwenden Exact Match
und Token-F1 aus dem Paket und implementieren BLEU, ROUGE, semantische Ähnlichkeit sowie
Faithfulness/Groundedness didaktisch dazu.

### Exact Match (EM)

$$\text{EM} = \frac{1}{|Q|}\sum_{q}\mathbb{1}[\text{norm}(\hat a_q) = \text{norm}(a_q)]$$

Die strengste Metrik: 1 nur bei exakter (normalisierter) Übereinstimmung. Nützlich für
kurze Faktoidantworten, irreführend bei freier Formulierung.

### Token-F1

$$P = \frac{|\hat T \cap T|}{|\hat T|},\quad R = \frac{|\hat T \cap T|}{|T|},\quad
F_1 = \frac{2PR}{P+R}$$

mit $\hat T, T$ als Token-Multimengen von Vorhersage und Referenz. Belohnt
Wortüberlappung, ignoriert Reihenfolge und Semantik.

### BLEU (vereinfacht)

$$\text{BLEU} = \text{BP}\cdot\exp\!\Big(\sum_{n=1}^{N} w_n \log p_n\Big)$$

mit modifizierter n-gramm-Präzision $p_n$ und Brevity Penalty BP gegen zu kurze Antworten.
Aus der maschinellen Übersetzung; präzisionsorientiert.

### ROUGE-1 / ROUGE-L

ROUGE-1 ist die Unigramm-Überlappung (recall-orientiert); ROUGE-L nutzt die längste
gemeinsame Teilsequenz (LCS) und erfasst so etwas Reihenfolge:

$$\text{ROUGE-L}_{F} = \frac{(1+\beta^2)\,R_{lcs}P_{lcs}}{R_{lcs} + \beta^2 P_{lcs}}$$

### Semantische Ähnlichkeit

Cosine-Ähnlichkeit der Embeddings von Vorhersage und Referenz. **Die einzige dieser
Metriken, die Bedeutung statt Oberfläche misst**. „7B Parameter" und „sieben Milliarden
Parameter" sind oberflächlich verschieden, semantisch identisch. Produktiv nutzt man dafür
den echten Embedder; ohne Modell verwenden wir hier einen lexikalischen Proxy und benennen
das ehrlich.

### Faithfulness und Groundedness

**Faithfulness:** Widerspricht die Antwort dem Kontext? **Groundedness:** Ist jede
Aussage der Antwort durch den Kontext *gestützt*? Beide adressieren Halluzination direkt.
Robuste Messung erfolgt produktiv per LLM-as-a-Judge (mit dessen Grenzen, Abschnitt 8);
hier approximieren wir Groundedness als Token-Überdeckung der Antwort durch den Kontext.

In [ ]:
from rag.evaluation.metrics.exact_match import ExactMatchMetric
from rag.evaluation.metrics.token_f1 import TokenF1Metric
# Faithfulness/Groundedness-Proxy ist Teil des Pakets: context_overlap misst den Anteil
# der Antwort-Inhaltstokens, die im Kontext verankert sind. Wir adaptieren nur die
# Signatur (ein Kontext-String -> Liste) und implementieren nichts neu.
from rag.evaluation.metrics.functional import context_overlap

def groundedness(answer: str, context: str) -> float:
    return context_overlap(answer, [context])

# --- Inline didactic implementations of metrics NOT in the package (BLEU/ROUGE/SemSim) ---
def _ngrams(tokens, n):
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)) if len(tokens) >= n else Counter()

def bleu(pred: str, ref: str, max_n: int = 2) -> float:
    p_toks, r_toks = tokenize(pred), tokenize(ref)
    if not p_toks:
        return 0.0
    precisions = []
    for n in range(1, max_n+1):
        pc, rc = _ngrams(p_toks, n), _ngrams(r_toks, n)
        if not pc:
            precisions.append(0.0); continue
        overlap = sum(min(c, rc[g]) for g, c in pc.items())
        precisions.append(overlap / max(sum(pc.values()), 1))
    if min(precisions) == 0:
        geo = 0.0
    else:
        geo = math.exp(sum(math.log(p) for p in precisions) / len(precisions))
    bp = 1.0 if len(p_toks) > len(r_toks) else math.exp(1 - len(r_toks)/max(len(p_toks), 1))
    return bp * geo

def rouge1(pred: str, ref: str) -> float:
    p, r = Counter(tokenize(pred)), Counter(tokenize(ref))
    overlap = sum(min(c, r[w]) for w, c in p.items())
    rec = overlap / max(sum(r.values()), 1)
    prec = overlap / max(sum(p.values()), 1)
    return 0.0 if (prec+rec) == 0 else 2*prec*rec/(prec+rec)

def _lcs(a, b):
    dp = [[0]*(len(b)+1) for _ in range(len(a)+1)]
    for i in range(1, len(a)+1):
        for j in range(1, len(b)+1):
            dp[i][j] = dp[i-1][j-1]+1 if a[i-1]==b[j-1] else max(dp[i-1][j], dp[i][j-1])
    return dp[-1][-1]

def rouge_l(pred: str, ref: str, beta: float = 1.2) -> float:
    p, r = tokenize(pred), tokenize(ref)
    if not p or not r:
        return 0.0
    l = _lcs(p, r)
    prec, rec = l/len(p), l/len(r)
    if prec+rec == 0:
        return 0.0
    return (1+beta**2)*prec*rec / (rec + beta**2*prec)

def semantic_sim(pred: str, ref: str) -> float:
    # Lexical cosine proxy; production uses embedding cosine (rag.embedding).
    p, r = Counter(tokenize(pred)), Counter(tokenize(ref))
    dot = sum(p[w]*r[w] for w in p)
    norm = math.sqrt(sum(v*v for v in p.values()))*math.sqrt(sum(v*v for v in r.values()))
    return dot/norm if norm else 0.0

print("EM/Token-F1 und Groundedness aus dem Paket; BLEU/ROUGE/SemSim didaktisch inline.")

In [ ]:
# Controlled answers for q2 ("Welche zwei Parameter steuern BM25?"), gold = "k1 und b."
reference = "k1 und b."
context_q2 = CORPUS["doc-bm25"]
answer_variants = {
    "exakt korrekt":        "k1 und b.",
    "korrekt, umformuliert":"Die beiden Parameter sind k1 sowie b.",
    "teilweise korrekt":    "Der wichtigste Parameter ist k1.",
    "halluziniert":         "BM25 wird durch alpha und gamma gesteuert.",
}

em = ExactMatchMetric()
f1 = TokenF1Metric()

def gen_pred(answer):
    ex = EvaluationExample(query="q", expected_answer=reference, relevant_document_ids=["doc-bm25"])
    from rag.evaluation.types import GeneratedAnswer
    ga = GeneratedAnswer(text=answer, model="mock", strategy="mock",
                         latency_ms=0.0, prompt_chars=0, context_chars=len(context_q2))
    return EvaluationPrediction(example=ex, retrieved_contexts=[], generated_answer=ga)

print(f"Referenz: {reference!r}\n")
print(f"{'Variante':<24} {'EM':>4} {'F1':>5} {'BLEU':>5} {'R-1':>5} {'R-L':>5} {'SemSim':>7} {'Ground':>7}")
print("-" * 72)
for label, ans in answer_variants.items():
    pred = gen_pred(ans)
    e = em.evaluate([pred]).value
    f = f1.evaluate([pred]).value
    print(f"{label:<24} {e:>4.0f} {f:>5.2f} {bleu(ans, reference):>5.2f} "
          f"{rouge1(ans, reference):>5.2f} {rouge_l(ans, reference):>5.2f} "
          f"{semantic_sim(ans, reference):>7.2f} {groundedness(ans, context_q2):>7.2f}")

**Interpretation und warum manche Metriken irreführen.** Die exakt korrekte Antwort
erreicht überall hohe Werte. Die *umformulierte* korrekte Antwort fällt bei EM auf 0 und
bei BLEU/ROUGE deutlich ab, obwohl sie inhaltlich richtig ist. Das ist der Kernfehler von
Oberflächenmetriken: Sie bestrafen legitime Variation. Die *halluzinierte* Antwort ist
besonders lehrreich: BLEU/ROUGE/F1 können moderat ausfallen (gemeinsame Füllwörter wie
„wird", „durch"), aber **Groundedness** kollabiert, weil die genannten Begriffe (alpha,
gamma) nicht im Kontext stehen. Das illustriert, warum Faithfulness/Groundedness für die
Halluzinationserkennung unverzichtbar sind und n-gramm-Metriken hier versagen.

In [ ]:
labels = list(answer_variants.keys())
metric_names = ["EM", "Token-F1", "BLEU", "ROUGE-L", "SemSim", "Ground"]
matrix = []
for ans in answer_variants.values():
    pred = gen_pred(ans)
    matrix.append([
        em.evaluate([pred]).value, f1.evaluate([pred]).value, bleu(ans, reference),
        rouge_l(ans, reference), semantic_sim(ans, reference), groundedness(ans, context_q2),
    ])
matrix = np.array(matrix)

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(metric_names)); w = 0.2
colors = ["#16a34a", "#2563eb", "#f59e0b", "#dc2626"]
for i, label in enumerate(labels):
    ax.bar(x + (i-1.5)*w, matrix[i], w, label=label, color=colors[i])
ax.set_xticks(x); ax.set_xticklabels(metric_names)
ax.set_ylabel("Wert"); ax.set_ylim(0, 1.05)
ax.set_title("Generation-Metriken nach Antwortqualitaet")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

---

## 8. Deterministische vs. subjektive Evaluation & die Grenzen von LLM-as-a-Judge

**Deterministische Metriken** (EM, Token-F1, BLEU, ROUGE, MRR, nDCG) sind reproduzierbar,
billig und schnell. Dieselbe Eingabe liefert denselben Wert. Ihr Preis: Sie messen
Oberfläche, nicht Bedeutung, und korrelieren oft schlecht mit menschlichem Urteil bei
freier Formulierung.

**Subjektive Evaluation** (Menschen oder LLM-as-a-Judge) erfasst Bedeutung, Stil,
Hilfreichkeit, also Dinge, die kein n-gramm misst. Ihr Preis: Kosten, Inkonsistenz, fehlende
Reproduzierbarkeit.

### Grenzen von LLM-as-a-Judge

Ein LLM als Bewerter ist verlockend. Er „versteht" Bedeutung und skaliert besser als
Menschen. Aber er erbt systematische Schwächen, die man kennen muss:

- **Positions- und Reihenfolge-Bias:** Bei Paarvergleichen bevorzugen LLM-Richter oft die
  erste (oder letzte) Antwort, unabhängig vom Inhalt.
- **Längen-Bias:** Längere, ausführlichere Antworten werden tendenziell höher bewertet,
  auch wenn sie nicht besser sind.
- **Self-Preference:** Ein Modell bewertet Ausgaben des eigenen Modellfamilien-Stils höher.
- **Nicht-Determinismus:** Bei `temperature > 0` schwankt das Urteil; selbst bei 0 ist es
  über Modellversionen nicht stabil.
- **Anfälligkeit für Formulierung:** Kleine Änderungen am Bewertungs-Prompt verschieben die
  Scores spürbar.
- **Halluzinierte Begründungen:** Der Richter kann eine plausibel klingende, aber falsche
  Begründung für seinen Score erfinden.

Die seriöse Praxis: LLM-as-a-Judge **kalibrieren** (gegen menschliche Urteile auf einer
Stichprobe validieren), Position randomisieren, mehrere Läufe mitteln, und nie als
alleinige Wahrheit behandeln. In diesem Notebook bleiben wir bewusst bei deterministischen
und heuristischen Metriken. Sie sind reproduzierbar und ihre Grenzen sind *bekannt*, was
ein eigener Wert ist.

---

## 9. Performance-Metriken

Qualität ohne Effizienz ist akademisch. Der `EvaluationRunner` erfasst über den
`PerformanceMonitor` automatisch Stage-Timings und (optional) Ressourcen-Momentaufnahmen. Die
`LatencyMetric` liefert End-to-End- und Per-Stage-Latenzen, `ThroughputMetric` die
Beispiele/Sekunde, `MemoryUsageMetric` die Spitzen-RSS.

In [ ]:
lat = retrieval_run.metric_results.get("latency")
thr = retrieval_run.metric_results.get("throughput")

if lat:
    print("Latenz (LatencyMetric):")
    print(f"  mean end-to-end : {lat.value:.2f} ms")
    md_ = lat.metadata
    print(f"  mean retrieval  : {md_.get('mean_retrieval_ms')}")
    print(f"  n gemessen      : {md_.get('n_e2e_measured')}")
if thr:
    unit = thr.metadata.get("unit", "")
    if thr.value >= 1.0:
        print(f"\nDurchsatz: {thr.value:,.1f} {unit}")
    else:
        # The demo retriever is sub-millisecond fast; the rate would round to ~0
        # and look like a bug. Report it honestly as "too fast to time stably".
        print(f"\nDurchsatz: zu schnell fuer eine stabile Messung "
              f"(roh {thr.value:.4f} {unit})")

print()
print("Hinweis: Diese Latenzen sind die des Demonstrations-Retrievers (reiner CPU-")
print("Token-Overlap, daher sehr schnell). Mit echtem Dense-Retrieval und LLM-Generierung")
print("dominiert die Generierung die Latenz um Groessenordnungen (siehe GPU-Notebook,")
print("Bottleneck-Analyse).")

---

## 10. Fehler- und Verteilungsvisualisierung

Aggregierte Mittelwerte verstecken, *welche* Beispiele scheitern. Wir berechnen die
Metriken pro Beispiel und visualisieren sie als Heatmap (Beispiel × Metrik) sowie als
Boxplot der Verteilung. Die Heatmap ist das schärfste Diagnosewerkzeug: dunkle Zeilen sind
Fehlerfälle.

In [ ]:
from rag.evaluation.metrics.context_precision import ContextPrecisionMetric
from rag.evaluation.metrics.context_recall import ContextRecallMetric

# Build per-example predictions from the demo retriever (retrieval-only).
preds = []
for ex in DATASET:
    results = retriever.retrieve(ex.query, k=retrieval_config.top_k)
    ctxs = [RetrievedContext(document_id=r["document_id"], chunk_id=r["chunk_id"],
                             text=r["text"], score=r["score"], rank=i)
            for i, r in enumerate(results)]
    preds.append(EvaluationPrediction(example=ex, retrieved_contexts=ctxs))

cp = ContextPrecisionMetric().evaluate(preds)
cr = ContextRecallMetric().evaluate(preds)
mr = MRRMetric().evaluate(preds)
nd = NDCGMetric(k=retrieval_config.top_k).evaluate(preds)

per_metric = {
    "CtxPrec": cp.per_example, "CtxRecall": cr.per_example,
    "MRR": mr.per_example, "nDCG": nd.per_example,
}
ex_ids = [p.example.example_id for p in preds]
M = np.array([per_metric[m] for m in per_metric])  # metrics x examples

fig, ax = plt.subplots(figsize=(9, 3.2))
im = ax.imshow(M, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
ax.set_yticks(range(len(per_metric))); ax.set_yticklabels(list(per_metric.keys()))
ax.set_xticks(range(len(ex_ids))); ax.set_xticklabels(ex_ids)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=7,
                color="black")
ax.set_title("Retrieval-Metriken pro Beispiel (gruen=gut, rot=Failure)")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

In [ ]:
# Boxplot of the per-example metric distributions.
fig, ax = plt.subplots(figsize=(7.5, 4))
data = [per_metric[m] for m in per_metric]
bp = ax.boxplot(data, labels=list(per_metric.keys()), patch_artist=True, showmeans=True)
for patch in bp["boxes"]:
    patch.set_facecolor("#bfdbfe")
ax.set_ylabel("Wert"); ax.set_ylim(-0.05, 1.05)
ax.set_title("Verteilung der Retrieval-Metriken ueber den Benchmark")
plt.tight_layout(); plt.show()
print("Eine breite Box / niedriger Median signalisiert instabile Retrieval-Qualitaet")
print("ueber die Queries hinweg - ein Hinweis, einzelne Failure Cases zu untersuchen.")

---

## 11. Halluzinationsanalyse

Halluzination ist das gefährlichste Fehlermuster, weil sie *flüssig* daherkommt. Wir
demonstrieren systematisch, dass Oberflächenmetriken sie schlecht erkennen und
Groundedness sie zuverlässig markiert. Dazu konstruieren wir für eine Query drei Antworten:
gestützt, ungestützt-aber-plausibel und ungestützt-falsch.

In [ ]:
hallu_query = "Was bewertet nDCG?"
hallu_context = CORPUS["doc-ndcg"]
hallu_reference = "Die gesamte Rangliste mit logarithmischem Discount."

cases = {
    "gestuetzt (korrekt)":
        "nDCG bewertet die gesamte Rangliste und gewichtet fruehe Positionen ueber einen logarithmischen Discount.",
    "fluessig, aber ungestuetzt":
        "nDCG wurde 1999 von Google eingefuehrt und basiert auf einem patentierten Klick-Modell.",
    "selbstbewusst falsch":
        "nDCG misst ausschliesslich die Antwortlatenz eines Systems in Millisekunden.",
}

print(f"Kontext (gekuerzt): {hallu_context[:90]}...\n")
print(f"{'Antwort':<32} {'BLEU':>5} {'ROUGE-L':>8} {'SemSim':>7} {'Ground':>7}")
print("-" * 64)
for label, ans in cases.items():
    print(f"{label:<32} {bleu(ans, hallu_reference):>5.2f} {rouge_l(ans, hallu_reference):>8.2f} "
          f"{semantic_sim(ans, hallu_reference):>7.2f} {groundedness(ans, hallu_context):>7.2f}")
print()
print("Beobachtung: Die fluessige Halluzination kann bei BLEU/ROUGE durchaus Punkte")
print("sammeln. Groundedness faellt jedoch deutlich, weil ihre Schluesselbegriffe")
print("(Google, 1999, Klick-Modell / Latenz, Millisekunden) im Kontext fehlen.")

**Interpretation.** Dies ist der empirische Kern der Halluzinationsproblematik: Eine
Antwort kann lexikalisch teilweise überlappen und trotzdem faktisch falsch sein.
**Groundedness**, der Anteil der Antwortbegriffe, die im Kontext verankert sind, trennt
gestützte von ungestützten Aussagen, wo n-gramm-Metriken blind sind. In Produktion ersetzt
man diese Heuristik durch ein LLM-as-a-Judge-Faithfulness-Urteil (mit den Vorbehalten aus
Abschnitt 8), aber das Prinzip bleibt: Halluzination misst man gegen den **Kontext**, nicht
gegen die Referenzformulierung.

---

## 12. Fehlerfälle identifizieren

Der in der Praxis wertvollste Schritt jeder Evaluation: Nicht der Mittelwert, sondern die
*schlechtesten* Beispiele zeigen, was zu reparieren ist. Wir sortieren die Beispiele nach
ihrem Reciprocal Rank und inspizieren die Fälle, in denen das relevante Dokument schlecht
oder gar nicht gefunden wurde.

In [ ]:
ranked = sorted(zip(ex_ids, mr.per_example, preds), key=lambda t: t[1])

print("Beispiele, aufsteigend nach Reciprocal Rank (schlechteste zuerst):")
print()
for eid, rr, pred in ranked:
    rel = set(pred.example.relevant_document_ids)
    got = [c.document_id for c in pred.retrieved_contexts]
    rank_of_rel = next((i+1 for i, d in enumerate(got) if d in rel), None)
    tag = "FAILURE" if rr < 0.5 else "ok"
    print(f"[{tag:<7}] {eid}  RR={rr:.3f}")
    print(f"          query   : {pred.example.query}")
    print(f"          relevant: {sorted(rel)}  | gefunden auf Rang: {rank_of_rel}")
    print(f"          top-3   : {got[:3]}")
    print()

**Interpretation.** Jeder Fehlerfall erzählt eine Ursachengeschichte. Steht das
relevante Dokument auf einem schlechten Rang, ist es ein *Ranking*-Problem (Reranking,
bessere Fusion könnten helfen). Fehlt es ganz unter den Top-k, ist es ein *Recall*-Problem
(größeres k, besserer Embedder, andere Chunking-Strategie). Beim Token-Overlap-Retriever
scheitern besonders Queries, deren Wortwahl von der des relevanten Dokuments abweicht,
genau die Schwäche, die dichtes Retrieval (Notebook 02/04) adressiert. Die Fehleranalyse
verbindet die Evaluation damit zurück an konkrete Architekturentscheidungen der Pipeline.

---

## 13. Reproduzierbarkeit, Benchmarking-Probleme und die Ground-Truth-Frage

### Reproduzierbarkeit

Eine Evaluation ist nur dann eine Messung, wenn sie reproduzierbar ist. Bedrohungen:
nicht-deterministische LLM-Inferenz (`temperature > 0`, sogar bei 0 über Versionen),
ungeseedete Stichproben, sich ändernde Modellgewichte, gleitende Korpora. Gegenmittel:
feste Seeds, versionierte Datensätze und Modellversionen (das Paket kodiert
`model_version` bereits in der deterministischen `embedding_id`), greedy decoding für
Faktoid-Evaluation, und Festhalten *aller* Konfigurationsparameter im Ergebnis
(`EvaluationRunResult.config_dict`).

### Benchmarking-Probleme

- **Zu kleine Stichproben.** Bei neun Beispielen schwankt jede Metrik dramatisch; ein
  einziger Fehler verschiebt den Mittelwert um gut 11 %. Der `EvaluationReport` warnt unter
  zehn Beispielen, der `BenchmarkReport` unter zwanzig Iterationen, aus genau diesem Grund.
- **Test-Set-Kontamination.** Wenn der Benchmark Teil der Trainingsdaten eines Modells war,
  misst man Memorisierung, nicht Fähigkeit.
- **Metrik-Gaming.** Optimiert man auf eine einzelne Metrik, optimiert man oft deren blinden
  Fleck mit. Mehrere komplementäre Metriken schützen davor.
- **Varianz vs. Effekt.** Ein Unterschied von 0,01 im nDCG zweier Systeme ist bedeutungslos,
  wenn die Lauf-zu-Lauf-Varianz größer ist. Der `BenchmarkReport` warnt bei hoher Varianz.

### Die Ground-Truth-Problematik

Jede Retrieval-Metrik setzt voraus, dass wir *wissen*, welche Dokumente relevant sind.
Diese Annahme ist fragil:

- **Relevanz ist graduell und subjektiv.** Unser Benchmark behandelt Relevanz binär (ein
  Dokument ist relevant oder nicht), aber real ist sie ein Spektrum.
- **Unvollständige Annotation.** Wird ein nicht annotiertes, aber tatsächlich relevantes
  Dokument zurückgegeben, wertet die Metrik es fälschlich als Fehler (Pooling-Bias).
- **Annotator-Drift.** Verschiedene Annotatoren, und derselbe Annotator zu verschiedenen
  Zeiten, urteilen unterschiedlich.

Die ehrliche Schlussfolgerung: Metriken sind Annäherungen an eine im Kern unscharfe
Wahrheit. Sie sind nützlich für *relative* Vergleiche („System A schlägt System B auf
diesem Datensatz") und gefährlich, wenn man sie als absolute Wahrheit liest.

---

## 14. Fazit

Dieses Notebook hat RAG-Evaluation als das behandelt, was sie ist: eine wissenschaftliche
Disziplin mit eigenen Methoden *und* eigenen blinden Flecken.

**Was gezeigt wurde:**

- Die **zwei Achsen**, Retrieval und Generation, müssen getrennt gemessen werden, sonst
  lässt sich eine schlechte Antwort keiner Ursache zuordnen (Abschnitt 4).
- **Retrieval-Metriken** (Precision@k, Recall@k, Hit Rate, MRR, nDCG, Context
  Precision/Recall) wurden formal definiert und an kontrollierten Ranglisten demonstriert
  (Abschnitt 6).
- **Generation-Metriken** (EM, Token-F1, BLEU, ROUGE, semantische Ähnlichkeit) wurden an
  vier Antwortqualitäten verglichen, mit dem zentralen Befund, dass Oberflächenmetriken
  legitime Umformulierungen bestrafen und Halluzination schlecht erkennen (Abschnitt 7).
- **Groundedness** trennt gestützte von halluzinierten Antworten, wo n-gramm-Metriken
  blind sind (Abschnitt 12).
- **Fehleranalyse** verbindet jeden schlechten Score zurück an eine konkrete
  Architekturursache (Ranking vs. Recall) und damit an die Pipeline-Entscheidungen der
  Notebooks 01 bis 05 (Abschnitt 13).
- **Reproduzierbarkeit, Benchmark-Größe und Ground-Truth-Unschärfe** wurden als die
  strukturellen Grenzen jeder Evaluation benannt (Abschnitt 14).

**Die methodische Kernbotschaft.** Es gibt keine einzelne Zahl für „RAG-Qualität". Es gibt
ein Bündel komplementärer Metriken, von denen jede einen Aspekt beleuchtet und jede einen
blinden Fleck hat. Professionelle Evaluation besteht nicht darin, eine Metrik zu maximieren,
sondern das Bündel zu verstehen, also zu wissen, *welche Metrik wann irreführt*. Genau diese
Skepsis ist in das `rag.evaluation`-Paket eingebaut: in die automatischen Warnungen bei zu
kleinen Stichproben, hoher Varianz und fehlenden Labels.

```text
                     Was diese Evaluation messen kann und was nicht
  Retrieval-Ranking      ████████████████  gut messbar (Ground Truth vorausgesetzt)
  Faktische Korrektheit  ████████          teils (EM/F1 streng, SemSim besser)
  Halluzination          ██████████        Groundedness/Faithfulness, nicht n-gramm
  Hilfreichkeit / Stil   ███               nur subjektiv / LLM-as-Judge (mit Grenzen)
  Reale Nutzerzufried.   ·                  nur online messbar
```
